In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import interpolate
from scipy.stats import gaussian_kde
import glob
import pickle
import matplotlib
import os

matplotlib.rcParams['font.size'] = 8.0
matplotlib.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'

In [16]:
def fit_nld_function(bins, values):
    # converts discrete nld to smoothened function which can be used for prediction
    
    import numpy as np
    from scipy.ndimage import gaussian_filter1d
    from scipy.interpolate import UnivariateSpline
#     from scipy.interpolate import PchipInterpolator

    # x: bin centers (projection values)
    # y: spike probability per bin
    x = np.array(bins[:-1])
    y = np.array(values)

    # --- Step 1: Sort (important for spline) ---
    idx = np.argsort(x)
    x = x[idx]
    y = y[idx]

    # --- Step 2: Smooth the discrete NLD ---
    y_smooth = gaussian_filter1d(y, sigma=1.5)  
    # sigma controls smoothing

    # --- Step 3: Fit spline ---
    spline = UnivariateSpline(x, y_smooth, s=0.001) # default cubic spline
    # s = smoothing factor (smaller = closer fit)
    
    # Monotonic spline
#     pchip = PchipInterpolator(x, y_smooth) # piecewise cubic spline
    
    return(spline)
#     # --- Step 4: Predict on new inputs ---
#     x_new = np.linspace(min(x), max(x), 500)
#     y_pred = spline(x_new)

In [17]:
def predictSpikeProbability(novel_stim, sta, nld_func, fs, prior_std, bin_center):
    # this function predicts spiking probability for a novel stimulus 
    # using prior constructed stan(mean position subtracted) and nld (in firing rate unit)
    
    sta_length = len(sta)
    total_stimulus_length = len(novel_stim)
    
    proj = np.full(shape=len(novel_stim), fill_value = np.nan)
    pred = np.full(shape=len(novel_stim), fill_value = np.nan)
    spikes = np.full(shape=len(novel_stim), fill_value = np.nan)

    dt = sta_length/fs
    for strt_index in np.arange(sta_length,total_stimulus_length):
        sliding_stim = novel_stim[strt_index-sta_length:strt_index] # get segment of stimulus
        proj[strt_index] = np.dot(sta,sliding_stim) # convolve onto sta      
#         pred_rate[strt_index] = pred[strt_index] * dt #spiking prob, dt is the time sampled == sta length
#         spikes[strt_index] = np.random.rand(len(pred)) < pred  # Poisson spiking 
        proj[strt_index] = proj[strt_index]/prior_std
    
        if proj[strt_index] < np.min(bin_center):
            proj[strt_index] = np.min(bin_center)
        elif proj[strt_index] > np.max(bin_center):
            proj[strt_index] = np.max(bin_center)
    
    #pred = nld_func(proj) # the predictied firing
    
    #return(proj, pred)

    pred = nld_func(proj)
    
    return proj[sta_length:], pred[sta_length:] #cut the Nan's here same as the STA length of 600 samples
    

In [18]:
Fs          = 3e4    #data sampling rate from orig
FS_ORIG     = 100    # natural behaviour video sampling rate
TimeHistory = 0.02   # 20ms pre-spike window (have not added post at all)
L           = int(TimeHistory * Fs)  # 600 samples, only consider length before 0

out_path = "./dataFolder/6_NaturalBehavior/Clipped_test/v2/"
os.makedirs(out_path, exist_ok=True)

In [19]:
# 1D model files 
model_files = glob.glob("./dataFolder/5_1DCompModel/Unscaled_clippedNLD/forForwardPrediction/v2/*.pickle")

In [20]:
# read in all the moth data

# Load the dictionary back from the file
with open('./dataFolder/6_NaturalBehavior/allmothMovement.pickle', 'rb') as file:
    moth_data = pickle.load(file)

print(f"data loaded from {len(moth_data)} with the following names {moth_data.keys()}")

data loaded from 8 with the following names dict_keys(['MV_m11_130802_errorAngle', 'MV_m13_130802_errorAngle', 'MV_m14_130806_errorAngle', 'MV_m1_130802_errorAngle', 'MV_m4_130802_errorAngle', 'MV_m5_130802_errorAngle', 'MV_m8_130805_errorAngle', 'MV_m9_130801_errorAngle'])


In [21]:
for filepath in model_files:
    with open(filepath, 'rb') as file:
        savedModel = pickle.load(file)
        
#     ste = savedModel[0]
    sta = savedModel[1]
#     sta_error  = savedModel[2]
#     prior_ensemble = savedModel[3]
#     prior_mean  = savedModel[4]
#     prior_error = savedModel[5]
    prior_std  = savedModel[6]
#     mean_fire_rate  = savedModel[7]
#     nld  = savedModel[8]
#     n_ste  = savedModel[9]
#     n_prior  = savedModel[10]
#     bin_centers  = savedModel[11]
    bin_fwdModel  = savedModel[12]
    nld_fwdModel  = savedModel[13]
    

    moth = os.path.basename(filepath).split('_')[0]
    unitID = "".join(os.path.basename(filepath).split('_')[1:3])
    print(f'working on {moth} {unitID}')

    nld_func = fit_nld_function(bin_fwdModel, nld_fwdModel)
    
#     # center the sta because it is offseted -# ?this step should have occured where we scale WN? - can't do it here
#     sta_mean = sta_mean - np.mean(sta_mean) 
    
    # take only the segment pre-Spike
    sta = sta[:L]    
    
#     # use this computational model for all moth videos
    for beh_moth, upsampled in list(moth_data.items()):
        vid_moth = "".join(beh_moth.split('_')[0:2]) 
        print(f'working on {vid_moth}')
        proj, p_spike = predictSpikeProbability(upsampled, sta, nld_func, Fs, prior_std, bin_fwdModel)
        spikes = np.random.rand(len(p_spike)) < p_spike

        # save data for future plotting and analysis
        saveObject = (nld_func, bin_fwdModel, nld_fwdModel,
                     proj, p_spike, spikes,
                     upsampled[len(sta):])
        with open(out_path + moth + '_' + unitID + '_for_' + vid_moth + "_Prediction.pickle","wb") as f:
            pickle.dump(saveObject, f)

working on m23 unit1
working on MVm11
working on MVm13
working on MVm14
working on MVm1
working on MVm4
working on MVm5
working on MVm8
working on MVm9
working on m29 unit1
working on MVm11
working on MVm13
working on MVm14
working on MVm1
working on MVm4
working on MVm5
working on MVm8
working on MVm9
working on m30 unit1
working on MVm11
working on MVm13
working on MVm14
working on MVm1
working on MVm4
working on MVm5
working on MVm8
working on MVm9
working on m31 unit1
working on MVm11
working on MVm13
working on MVm14
working on MVm1
working on MVm4
working on MVm5
working on MVm8
working on MVm9
working on m31 unit2
working on MVm11
working on MVm13
working on MVm14
working on MVm1
working on MVm4
working on MVm5
working on MVm8
working on MVm9
working on m31 unit3
working on MVm11
working on MVm13
working on MVm14
working on MVm1
working on MVm4
working on MVm5
working on MVm8
working on MVm9
